In [ ]:
import cv2
import os
import glob
import numpy as np
from collections import deque
from wisardpkg import KernelCanvas, ClusWisard
from wisard_object_tracker.src.utils import tracker_utils

# ==============================
# CONFIGURAÇÕES
# ==============================
TEMPORAL_WINDOW = 5
STEP = 10

KERNEL_DIM = 3
N_KERNELS = 128

CONFIDENCE_THRESHOLD = 0.6

# ==============================
# CARREGAR DADOS
# ==============================
DATASET_ROOT_FOLDER = '/mnt/c/Users/Isabella/TCC/wisard_object_tracker/data/david'
IMAGE_FOLDER = os.path.join(DATASET_ROOT_FOLDER, 'imgs')
GT_TXT_PATH = os.path.join(DATASET_ROOT_FOLDER, 'david_gt.txt')

image_paths = sorted(glob.glob(os.path.join(IMAGE_FOLDER, '*.png')))
frames = [cv2.imread(p) for p in image_paths]

all_ground_truths = tracker_utils.load_ground_truth_from_gt_txt(GT_TXT_PATH)

# ==============================
# MODELOS
# ==============================
kernel_canvas = KernelCanvas(
    KERNEL_DIM,
    N_KERNELS,
    bitsByKernel=7,
    activationDegree=0.45,
    useDirection=True
)

temporal_buffer = deque(maxlen=TEMPORAL_WINDOW)

patch = tracker_utils.extract_patch(frames[0],all_ground_truths[0])
temporal_buffer.append(patch_features(patch))

binary = kernel_canvas.transform(list(temporal_buffer))


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def visualize_kernel_canvas_output(binary, n_cols=32):
    binary = np.array(binary)

    # completa com zeros para formar matriz
    remainder = binary.size % n_cols
    if remainder != 0:
        padding = n_cols - remainder
        binary = np.pad(binary, (0, padding))

    grid = binary.reshape(-1, n_cols)

    plt.figure(figsize=(6, 4))
    plt.imshow(grid, cmap="gray")
    plt.colorbar(label="Activation (0/1)")
    plt.title("KernelCanvas binary activation map")
    plt.xlabel("Kernel index (grouped)")
    plt.ylabel("Activation blocks")
    plt.show()


In [ ]:
binary = kernel_canvas.transform(list(temporal_buffer))

visualize_kernel_canvas_output(binary)
